In [3]:
import pandas as pd

full = pd.read_excel("15000business_statements.xlsx")
manual = pd.read_excel("manual15000business_change_statements.xlsx")

change_qualifier_ids = {
    'P12413', 'P12506', 'P1264', 'P1319', 'P1326', 'P1365', 'P1366',
    'P156', 'P2754', 'P3415', 'P571', 'P576', 'P577', 'P580',
    'P582', 'P585', 'P7888', 'P8554', 'P8555'
}

# Important: business file uses Qualifier_Property_ID_1, Qualifier_Property_ID_2, ...
qual_id_cols = [c for c in full.columns if c.startswith("Qualifier_Property_ID")]

print("Qualifier columns found:", qual_id_cols[:5])
print("Number of qualifier columns:", len(qual_id_cols))

# Normalize IDs
full["QID"] = full["QID"].astype(str).str.strip()
full["Property_ID"] = full["Property_ID"].astype(str).str.strip()
manual["QID"] = manual["QID"].astype(str).str.strip()
manual["Property_ID"] = manual["Property_ID"].astype(str).str.strip()

# 1. Context-dependent properties identified in the final reviewed dataset
context_property_ids = set(
    manual.loc[manual["change_class"] == "class2", "Property_ID"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Number of context-dependent properties: {len(context_property_ids)}")

# 2. All full-sample pairs using these context-dependent properties
context_full = full[
    full["Property_ID"].isin(context_property_ids)
].copy()

# 3. Raw statement-level flag: whether a statement has a change-related qualifier
def has_change_qualifier(row):
    for col in qual_id_cols:
        val = row.get(col)
        if pd.notna(val):
            qid = str(val).strip()
            if qid in change_qualifier_ids:
                return True
    return False

context_full["has_change_qualifier_stmt"] = context_full.apply(
    has_change_qualifier, axis=1
)

# 4. Final reviewed class2 pair keys
manual_class2_pairs = (
    manual.loc[manual["change_class"] == "class2", ["QID", "Property_ID"]]
    .drop_duplicates()
    .copy()
)
manual_class2_pairs["reviewed_change_pair"] = True

# 5. Pair-level summary in the full sample
pair_summary = (
    context_full
    .groupby(["QID", "Property_ID"])
    .agg(
        statement_count=("Statement_Number", "nunique"),
        value_count=("Main_Value_Label", "nunique"),
        has_raw_change_qualifier=("has_change_qualifier_stmt", "any")
    )
    .reset_index()
)

# 6. Add whether the pair is in the final reviewed class2 dataset
pair_summary = pair_summary.merge(
    manual_class2_pairs,
    on=["QID", "Property_ID"],
    how="left"
)

pair_summary["reviewed_change_pair"] = pair_summary["reviewed_change_pair"].fillna(False)

# 7. Counts
total_pairs = len(pair_summary)

raw_with_qual = pair_summary["has_raw_change_qualifier"].sum()
raw_without_qual = total_pairs - raw_with_qual

reviewed_with_change = pair_summary["reviewed_change_pair"].sum()
reviewed_without_change = total_pairs - reviewed_with_change

reviewed_without_change_multi_value = pair_summary[
    (~pair_summary["reviewed_change_pair"]) &
    (pair_summary["value_count"] > 1)
]

print("\n=== Raw qualifier signal ===")
print(f"Total pairs using context-dependent properties: {total_pairs}")
print(f"Pairs with raw change-related qualifier: {raw_with_qual} ({raw_with_qual / total_pairs:.2%})")
print(f"Pairs without raw change-related qualifier: {raw_without_qual} ({raw_without_qual / total_pairs:.2%})")

print("\n=== Final reviewed signal ===")
print(f"Pairs identified as reviewed class2 change-related pairs: {reviewed_with_change} ({reviewed_with_change / total_pairs:.2%})")
print(f"Pairs not identified as reviewed class2 change-related pairs: {reviewed_without_change} ({reviewed_without_change / total_pairs:.2%})")
print(
    "Pairs not identified as reviewed class2 change-related pairs but with multiple values: "
    f"{len(reviewed_without_change_multi_value)} "
    f"({len(reviewed_without_change_multi_value) / total_pairs:.2%})"
)


pair_summary.to_excel(
    "0701context_dependent_property_pairs_qualifier_summary_business.xlsx",
    index=False
)

reviewed_without_change_multi_value.to_excel(
    "0701context_dependent_pairs_without_reviewed_change_multi_value_business.xlsx",
    index=False
)

print("\nSaved:")
print("0701context_dependent_property_pairs_qualifier_summary_business.xlsx")
print("0701context_dependent_pairs_without_reviewed_change_multi_value_business.xlsx")

Qualifier columns found: ['Qualifier_Property_ID_1', 'Qualifier_Property_ID_2', 'Qualifier_Property_ID_3', 'Qualifier_Property_ID_4', 'Qualifier_Property_ID_5']
Number of qualifier columns: 16
Number of context-dependent properties: 209

=== Raw qualifier signal ===
Total pairs using context-dependent properties: 85333
Pairs with raw change-related qualifier: 5598 (6.56%)
Pairs without raw change-related qualifier: 79735 (93.44%)

=== Final reviewed signal ===
Pairs identified as reviewed class2 change-related pairs: 5598 (6.56%)
Pairs not identified as reviewed class2 change-related pairs: 79735 (93.44%)
Pairs not identified as reviewed class2 change-related pairs but with multiple values: 5631 (6.60%)

Saved:
0701context_dependent_property_pairs_qualifier_summary_business.xlsx
0701context_dependent_pairs_without_reviewed_change_multi_value_business.xlsx
